In [1]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\DELL\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.2.0
c:\Users\DELL\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [4]:
train_df = pd.read_csv("C:/Users/DELL/Regression_ML_EndtoEnd/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("C:/Users/DELL/Regression_ML_EndtoEnd/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [6]:
import mlflow

# 1. Pointer exactement vers le même serveur/base SQLite que votre UI MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# 2. Définir le nom de l'expérience (utilisez experiment_name= pour éviter toute ambiguïté)
mlflow.set_experiment(experiment_name="xgboost_optuna_housing")

# 3. Lancer votre étude Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

[I 2026-10-04 11:50:12,428] A new study created in memory with name: no-name-104be24e-5131-4dd0-a3de-b0eec797b319
[I 2026-10-04 11:50:34,881] Trial 0 finished with value: 76173.28648136552 and parameters: {'n_estimators': 648, 'max_depth': 4, 'learning_rate': 0.04141434198166385, 'subsample': 0.7096365059800402, 'colsample_bytree': 0.866045120253489, 'min_child_weight': 7, 'gamma': 4.476807878926356, 'reg_alpha': 0.008499999572983615, 'reg_lambda': 0.3517816624366389}. Best is trial 0 with value: 76173.28648136552.
[I 2026-10-04 11:50:48,249] Trial 1 finished with value: 72731.20847888889 and parameters: {'n_estimators': 271, 'max_depth': 6, 'learning_rate': 0.07239427271314514, 'subsample': 0.5549954049971921, 'colsample_bytree': 0.6662509381361807, 'min_child_weight': 4, 'gamma': 0.08929515675296873, 'reg_alpha': 3.028701573832311e-06, 'reg_lambda': 0.021274531573753943}. Best is trial 1 with value: 72731.20847888889.
[I 2026-10-04 11:51:21,832] Trial 2 finished with value: 70355.772

Best params: {'n_estimators': 808, 'max_depth': 7, 'learning_rate': 0.030231797443126512, 'subsample': 0.8280406132142354, 'colsample_bytree': 0.6250641953051196, 'min_child_weight': 3, 'gamma': 2.9911355591474442, 'reg_alpha': 0.0002814561261709805, 'reg_lambda': 1.0610722408968293e-05}


In [7]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31375.187393452128
RMSE: 69806.41323653971
R²: 0.9623425698788718
